# DSAR × Lakeflow Declarative Pipelines · 00 · Setup & landing

Builds the **isolated demo schema**, the **`raw_user`** landing table the pipeline
reads, and a multi-subject **`dsar_request`** table (the erasure queue, mirroring
the base `dsar_erasure/` layer).

Pipeline topology this feeds (a clean linear medallion):

```
raw_user  ─▶  bronze_user  ─▶  silver_user  ─▶  gold_user
 (landing)     (mask PII)       (cleaned)        (per-customer aggregate)
```

> Batch notebook — **Run all** once. Idempotent: drops & rebuilds the schema.
> All names are widget-driven; nothing is Allegiant-specific.

## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (isolated demo)")
dbutils.widgets.text("num_users", "2000", "3 Number of customers")
dbutils.widgets.text("events_per_user", "5", "4 Raw events per customer")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
N_USERS = int(dbutils.widgets.get("num_users"))
N_EV    = int(dbutils.widgets.get("events_per_user"))
print("Target schema:", FQ, "| customers:", N_USERS, "| raw events:", N_USERS * N_EV)

## 1. Clean rebuild of the demo schema

The pipeline publishes `bronze_user`/`silver_user`/`gold_user` into this same
schema at runtime.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")
spark.sql(f"CREATE SCHEMA {FQ} COMMENT 'DSAR erasure integrated with a Lakeflow Declarative Pipeline'")
print("Recreated", FQ)

## 2. `raw_user` — the landing table (what Auto Loader would write)

- `user_id` — **stable business key**, never PII, never masked. The whole pipeline
  keys on it, so a subject still resolves *after* PII is masked.
- `email`, `full_name` — **PII**, masked by bronze.
- `profile_json` — nested blob with `contact.email` + `contact.name` PII (the
  in-JSON masking case) + non-PII `loyalty`.
- `revenue`, `event_ts`, `_ingest_ts` — non-PII payload, **preserved**.

Plain append-only Delta table.

In [ ]:
from pyspark.sql import functions as F

FIRST = ["Alex","Sam","Jordan","Taylor","Morgan","Casey","Riley","Jamie","Drew","Quinn"]
LAST  = ["Lucero","Ortiz","Nguyen","Patel","Kim","Diaz","Reed","Cole","Shah","Vega"]

base = (
    spark.range(N_USERS)
    .withColumn("user_id", F.concat(F.lit("U"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("first", F.element_at(F.array(*[F.lit(x) for x in FIRST]), (F.col("id") % 10 + 1).cast("int")))
    .withColumn("last",  F.element_at(F.array(*[F.lit(x) for x in LAST]),  (F.col("id") % 10 + 1).cast("int")))
    .withColumn("full_name", F.concat_ws(" ", "first", "last"))
    .withColumn("email", F.concat_ws("", F.lower("first"), F.lit("."), F.lower("last"),
                                     F.col("id").cast("string"), F.lit("@example.com")))
)
events = (
    base.withColumn("evt", F.explode(F.sequence(F.lit(0), F.lit(N_EV - 1))))
    .withColumn("event_id", F.concat_ws("-", "user_id", F.col("evt").cast("string")))
    .withColumn("revenue", F.round(F.rand(7) * 500, 2))
    .withColumn("event_ts", F.expr("current_timestamp() - make_interval(0,0,0,cast(evt as int),0,0,0)"))
    # per-event ingest time (evt grows => later) so any SCD1 sequencing is deterministic
    .withColumn("_ingest_ts", F.expr("current_timestamp() + make_interval(0,0,0,0,0,cast(evt as int),0)"))
    .withColumn("profile_json", F.to_json(F.struct(
        F.struct(F.col("email").alias("email"), F.col("full_name").alias("name")).alias("contact"),
        F.struct(F.lit("gold").alias("tier"), F.col("revenue").alias("ltv")).alias("loyalty"))))
    .select("event_id", "user_id", "email", "full_name", "profile_json",
            "revenue", "event_ts", "_ingest_ts")
)
events.write.mode("overwrite").saveAsTable(f"{FQ}.raw_user")
print("raw_user rows:", spark.table(f"{FQ}.raw_user").count())
display(spark.table(f"{FQ}.raw_user").limit(5))

## 3. `dsar_request` — the erasure queue (multi-subject)

Mirrors the base `dsar_erasure/` layer: each row is one DSAR request naming one
subject by **email**, with a `request_type` (DELETE or OBFUSCATE) and a `status`.
Notebook `02` processes all PENDING rows in one run — erasing every named subject
across every layer.

In [ ]:
from pyspark.sql import functions as F

# pick a few real subjects from raw_user so the requests target real data
subjects = (spark.table(f"{FQ}.raw_user").select("email").distinct()
            .orderBy("email").limit(3).collect())
emails = [r["email"] for r in subjects]

rows = [
    (f"REQ-{i+1:03d}", e, ("DELETE" if i % 2 == 0 else "OBFUSCATE"), "PENDING")
    for i, e in enumerate(emails)
]
df = spark.createDataFrame(rows, "request_id string, subject_email string, request_type string, status string") \
          .withColumn("request_date", F.current_date()) \
          .withColumn("deadline_date", F.date_add(F.current_date(), 45))
df.write.mode("overwrite").saveAsTable(f"{FQ}.dsar_request")

print("Seeded DSAR requests:")
display(spark.table(f"{FQ}.dsar_request"))
print("\nDemo subjects (used by 02):", emails)

## 4. Next

1. **`01_sdp_pipeline`** — attach as a Lakeflow pipeline source (clean-silver
   variant). Or **`01b_sdp_pipeline_cdc_variant`** for the SCD1 silver.
2. **`02_erasure`** — process the DSAR queue: erase every subject at every layer.